# GCascadeV5 Tutorial

This notebook introduces the current Python port of GCascade (`GCascadeV5`).

It assumes the code is in `/Users/antonio/Desktop/Research/GCascadeV5` and the V4 libraries are available at `/Users/antonio/Desktop/Research/GCascade/LibrariesV4`.

## 1) Environment setup

In [ ]:
# Optional: set the read-only table location explicitly
import os
os.environ['GCASCADE_LIB_PATH'] = '/Users/antonio/Desktop/Research/GCascade/LibrariesV4'

In [ ]:
import numpy as np
import gcascade_v5 as gc

print('energies shape:', gc.energies.shape)
print('diffuseDistances shape:', gc.diffuseDistances.shape)
print('current EBL index:', gc.EBLindex)

## 2) Build an injected spectrum

In [ ]:
inj = gc.cutoffPowerLaw(gc.energies, gamma=2.2, cutoff=1e7, amp=1e40)
print('inj min/max:', inj.min(), inj.max())

## 3) Point-source propagation

In [ ]:
z_source = 0.3

phi_redshift = gc.RedshiftPoint(inj, z_source)
phi_atten = gc.AttenuatePoint(inj, z_source)
phi_cascade = gc.CascadePoint(inj, z_source)

print(phi_redshift.shape, phi_atten.shape, phi_cascade.shape)

## 4) Diffuse population

In [ ]:
# Example comoving source density profile
z = gc.diffuseDistances
z_distrib = 1e-9 * np.exp(-((z - 1.0) / 0.7) ** 2)

phi_diff_redshift = gc.RedshiftDiffuse(inj, 3.0, z_distrib)
phi_diff_atten = gc.AttenuateDiffuse(inj, 3.0, z_distrib)
phi_diff_cascade = gc.CascadeDiffuse(inj, 3.0, z_distrib)

print(phi_diff_redshift.shape, phi_diff_atten.shape, phi_diff_cascade.shape)

## 5) Evolving source spectra

In [ ]:
# Simple evolving injection: softening spectrum with redshift
inj2d = np.empty((len(gc.diffuseDistances), len(gc.energies)))
for i, zi in enumerate(gc.diffuseDistances):
    gamma_i = 2.0 + 0.2 * zi / 10.0
    inj2d[i] = gc.cutoffPowerLaw(gc.energies, gamma=gamma_i, cutoff=1e7, amp=1e40)

phi_ev_redshift = gc.RedshiftEvolving(inj2d, 3.0, z_distrib)
phi_ev_atten = gc.AttenuateEvolving(inj2d, 3.0, z_distrib)
phi_ev_cascade = gc.CascadeEvolving(inj2d, 3.0, z_distrib)

print(phi_ev_redshift.shape, phi_ev_atten.shape, phi_ev_cascade.shape)

## 6) Switch EBL model

In [ ]:
# EBL indices: 0 CMB-only, 1 SL, 2 SLhigh, 3 SLlow, 4 Finke, 5 Franc, 6 Dom
old_ebl = gc.EBLindex
if old_ebl != 0:
    gc.changeEBLModel(0)
print('current EBL index:', gc.EBLindex)

# switch back
if gc.EBLindex != old_ebl:
    gc.changeEBLModel(old_ebl)
print('restored EBL index:', gc.EBLindex)

## 7) Optional plotting

In [ ]:
fig, ax = gc.specPlot(phi_cascade)
ax.set_title('E^2 Phi(E) for CascadePoint example')

## 8) Parity workflow with V4

1. Export V4 reference fixtures using `scripts/export_v4_reference_template.wl`.
2. Place fixture directories under `benchmarks/v4_reference/`.
3. Run:

```bash
python3 scripts/run_parity.py
```

A case passes if max relative error is below the configured tolerance (default `1e-3`).